# Note 01 Deep Dive: Linear Systems, Determinants & Inverses

Companion to **Note 01** (Applied Math for DS & AI). Every worked example and case study in the note is verified here.

| Part | Topic |
|---|---|
| 1 | Determinants: cofactor expansion, Sarrus, properties |
| 2 | Inverse via the adjugate; (AB)⁻¹ = B⁻¹A⁻¹ |
| 3 | Cramer's rule |
| 4 | Orthogonal matrices: rotations, reflections, numerical drift |
| 5 | Hill cipher (invertibility mod 26) |
| 6 | Least squares: normal equations, line fit, GPS-style positioning |
| 7 | Conditioning: when tiny input changes cause huge output changes |
| 8 | Diet problem as a 3×3 linear system |
| 9 | Leontief input–output model and the Hill-cipher known-plaintext attack |
| 10 | Checks for the practice problems in the note |


In [1]:
import numpy as np
import sympy as sp
np.set_printoptions(precision=4, suppress=True)

## Part 1 — Determinants

In [2]:
def det_cofactor(A):
    # Laplace (cofactor) expansion along the first row: O(n!) work.
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    if n == 1:
        return A[0, 0]
    if n == 2:
        return A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    total = 0.0
    for j in range(n):
        minor = np.delete(np.delete(A, 0, axis=0), j, axis=1)
        total += (-1) ** j * A[0, j] * det_cofactor(minor)
    return total

def det_sarrus(A):
    # Rule of Sarrus: valid ONLY for 3x3.
    (a, b, c), (d, e, f), (g, h, i) = A
    plus = a*e*i + b*f*g + c*d*h
    minus = c*e*g + a*f*h + b*d*i
    return plus - minus, plus, minus

A = np.array([[2, 1, 3], [0, -1, 4], [1, 2, 0]])
print("cofactor expansion:", det_cofactor(A))
print("Sarrus (det, +diagonals, -diagonals):", det_sarrus(A))
print("numpy:", round(np.linalg.det(A), 6))

cofactor expansion: -9.0
Sarrus (det, +diagonals, -diagonals): (np.int64(-9), np.int64(4), np.int64(13))
numpy: -9.0


In [3]:
# Properties of the determinant, checked on random integer matrices
rng = np.random.default_rng(0)
P = rng.integers(-5, 6, size=(4, 4)).astype(float)
Q = rng.integers(-5, 6, size=(4, 4)).astype(float)
d = np.linalg.det
print("det(PQ) =", round(d(P @ Q), 4), "  det(P)det(Q) =", round(d(P) * d(Q), 4))
print("det(Pᵀ) =", round(d(P.T), 4), "  det(P) =", round(d(P), 4))
print("det(3P) =", round(d(3 * P), 4), "  3⁴ det(P) =", round(3**4 * d(P), 4))
Ps = P[[1, 0, 2, 3]]                     # swap rows 1 and 2
print("row swap      :", round(d(Ps), 4), "(sign flips)")
Pr = P.copy(); Pr[2] = Pr[2] + 7 * Pr[0]  # R3 <- R3 + 7 R1
print("R3 += 7 R1    :", round(d(Pr), 4), "(unchanged)")
U = np.triu(P)
print("triangular det:", round(d(U), 4), " product of diagonal:", np.prod(np.diag(U)))

det(PQ) = -49320.0   det(P)det(Q) = -49320.0
det(Pᵀ) = -274.0   det(P) = -274.0
det(3P) = -22194.0   3⁴ det(P) = -22194.0
row swap      : 274.0 (sign flips)
R3 += 7 R1    : -274.0 (unchanged)
triangular det: -120.0  product of diagonal: -120.0


In [4]:
# Symbolic proof for 2x2: det(AB) = det(A) det(B)
a, b, c, e, p, q, r, s = sp.symbols('a b c e p q r s')
A2 = sp.Matrix([[a, b], [c, e]]); B2 = sp.Matrix([[p, q], [r, s]])
print("det(AB) - det(A)det(B) simplifies to:", sp.expand((A2 * B2).det() - A2.det() * B2.det()))

det(AB) - det(A)det(B) simplifies to: 0


## Part 2 — Inverse via the adjugate

In [5]:
def adjugate(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    Cof = np.zeros_like(A)
    for i in range(n):
        for j in range(n):
            minor = np.delete(np.delete(A, i, axis=0), j, axis=1)
            Cof[i, j] = (-1) ** (i + j) * det_cofactor(minor)
    return Cof.T          # adjugate = transpose of the cofactor matrix

B = np.array([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
adjB = adjugate(B)
print("det(B) =", det_cofactor(B))
print("Adj(B) =\n", adjB)
print("B @ Adj(B) = det(B) I:\n", B @ adjB)
print("B⁻¹ = Adj/det =\n", adjB / det_cofactor(B))

det(B) = 1.0
Adj(B) =
 [[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]
B @ Adj(B) = det(B) I:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
B⁻¹ = Adj/det =
 [[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]


In [6]:
# (AB)⁻¹ = B⁻¹A⁻¹  (order reverses!)
A = np.array([[1, 2], [3, 5]]); B = np.array([[2, 1], [1, 1]])
inv = np.linalg.inv
print("(AB)⁻¹ =\n", inv(A @ B))
print("B⁻¹A⁻¹ =\n", inv(B) @ inv(A))
print("A⁻¹B⁻¹ (wrong order) =\n", inv(A) @ inv(B))

(AB)⁻¹ =
 [[-8.  3.]
 [11. -4.]]
B⁻¹A⁻¹ =
 [[-8.  3.]
 [11. -4.]]
A⁻¹B⁻¹ (wrong order) =
 [[-7.  9.]
 [ 4. -5.]]


## Part 3 — Cramer's rule

In [7]:
def cramer(A, b):
    A = np.asarray(A, dtype=float); D = det_cofactor(A)
    if abs(D) < 1e-12:
        raise ValueError("det(A) = 0: Cramer's rule does not apply")
    xs, Ds = [], []
    for i in range(A.shape[1]):
        Ai = A.copy(); Ai[:, i] = b
        Ds.append(det_cofactor(Ai)); xs.append(Ds[-1] / D)
    return np.array(xs), D, Ds

A = np.array([[1, 1, 1], [0, 2, 5], [2, 5, -1]]); b = np.array([6, -4, 27])
x, D, Ds = cramer(A, b)
print("D =", D, " D_i =", Ds, " x =", x, " check Ax =", A @ x)

D = -21.0  D_i = [np.float64(-105.0), np.float64(-63.0), np.float64(42.0)]  x = [ 5.  3. -2.]  check Ax = [ 6. -4. 27.]


In [8]:
# Why nobody uses Cramer/cofactors for big n: count the time
import time
for n in [6, 7, 8]:
    M = rng.standard_normal((n, n)); t = time.perf_counter(); det_cofactor(M)
    t1 = time.perf_counter() - t; t = time.perf_counter(); np.linalg.det(M); t2 = time.perf_counter() - t
    print(f"n={n}: cofactor expansion {t1*1e3:9.2f} ms   LU (numpy) {t2*1e3:.3f} ms")

n=6: cofactor expansion      6.26 ms   LU (numpy) 0.044 ms
n=7: cofactor expansion     33.12 ms   LU (numpy) 0.046 ms


n=8: cofactor expansion    199.66 ms   LU (numpy) 0.041 ms


## Part 4 — Orthogonal matrices: rotations and reflections

In [9]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

R = rot(np.deg2rad(60))
print("R(60°) =\n", R)
print("RᵀR =\n", R.T @ R, "\ndet R =", round(np.linalg.det(R), 6))
v = np.array([3.0, 4.0])
print("‖v‖ =", np.linalg.norm(v), " ‖Rv‖ =", round(np.linalg.norm(R @ v), 10))
print("R(30°)R(45°) == R(75°)?", np.allclose(rot(np.deg2rad(30)) @ rot(np.deg2rad(45)), rot(np.deg2rad(75))))

R(60°) =
 [[ 0.5   -0.866]
 [ 0.866  0.5  ]]
RᵀR =
 [[ 1. -0.]
 [-0.  1.]] 
det R = 1.0
‖v‖ = 5.0  ‖Rv‖ = 5.0
R(30°)R(45°) == R(75°)? True


In [10]:
# Reflection about the line at angle θ: F = I - 2 n nᵀ with n the unit normal; F² = I
th = np.deg2rad(30)
n = np.array([-np.sin(th), np.cos(th)])
F = np.eye(2) - 2 * np.outer(n, n)
print("F =\n", F, "\nF² =\n", F @ F, "\ndet F =", round(np.linalg.det(F), 6))
print("direction of mirror is fixed:", F @ np.array([np.cos(th), np.sin(th)]))

F =
 [[ 0.5    0.866]
 [ 0.866 -0.5  ]] 
F² =
 [[ 1. -0.]
 [-0.  1.]] 
det F = -1.0
direction of mirror is fixed: [0.866 0.5  ]


In [11]:
# Robotics / graphics: compose 1° rotations 3600 times in float32 → numerical drift
R1 = rot(np.deg2rad(1)).astype(np.float32)
Racc = np.eye(2, dtype=np.float32)
for _ in range(3600):
    Racc = Racc @ R1
err = np.linalg.norm(Racc.T @ Racc - np.eye(2))
print("after 3600 steps (10 full turns): ‖RᵀR - I‖ =", err, " det =", np.linalg.det(Racc.astype(float)))
U, S, Vt = np.linalg.svd(Racc.astype(float))
Rfix = U @ Vt                          # nearest orthogonal matrix
print("after SVD re-orthogonalisation:   ‖RᵀR - I‖ =", np.linalg.norm(Rfix.T @ Rfix - np.eye(2)))

after 3600 steps (10 full turns): ‖RᵀR - I‖ = 0.0001227316228637234  det = 1.000086786251078
after SVD re-orthogonalisation:   ‖RᵀR - I‖ = 6.718007637568815e-23


## Part 5 — Hill cipher: encryption needs an invertible key mod 26

In [12]:
def to_nums(s):  return [ord(ch) - 65 for ch in s]
def to_text(v):  return "".join(chr(int(k) + 65) for k in v)

def hill(text, K):
    v = to_nums(text)
    out = []
    for i in range(0, len(v), 2):
        out += list((K @ np.array(v[i:i+2])) % 26)
    return to_text(out)

def inv_mod26(K):
    det = int(round(np.linalg.det(K))) % 26
    det_inv = pow(det, -1, 26)          # raises ValueError if gcd(det, 26) != 1
    a, b, c, d = K.ravel()
    adj = np.array([[d, -b], [-c, a]])
    return (det_inv * adj) % 26

K = np.array([[3, 3], [2, 5]])
Kinv = inv_mod26(K)
print("det K =", round(np.linalg.det(K)), "→ det⁻¹ mod 26 =", pow(9, -1, 26))
print("K⁻¹ mod 26 =\n", Kinv, "\nK K⁻¹ mod 26 =\n", (K @ Kinv) % 26)
c = hill("HELP", K); print("HELP →", c, "→", hill(c, Kinv))
msg = "LINEARALGEBRA"  + "X"            # pad to even length
c = hill(msg, K); print(msg, "→", c, "→", hill(c, Kinv))

det K = 9 → det⁻¹ mod 26 = 3
K⁻¹ mod 26 =
 [[15 17]
 [20  9]] 
K K⁻¹ mod 26 =
 [[1 0]
 [0 1]]
HELP → HIAT → HELP
LINEARALGEBRAX → FKZUZHHDEGCJRL → LINEARALGEBRAX


In [13]:
# A key with even determinant cannot be inverted mod 26
Kbad = np.array([[2, 4], [1, 3]])
print("det =", round(np.linalg.det(Kbad)))
try:
    inv_mod26(Kbad)
except ValueError as e:
    print("ValueError:", e)
print("Two different plaintexts, same ciphertext:", hill("AN", Kbad), hill("NA", Kbad))

det = 2
ValueError: base is not invertible for the given modulus
Two different plaintexts, same ciphertext: AN AN


## Part 6 — Least squares and the normal equations

In [14]:
x = np.array([0, 1, 2, 3.]); y = np.array([1, 3, 4, 4.])
X = np.c_[np.ones_like(x), x]
print("XᵀX =\n", X.T @ X, "\nXᵀy =", X.T @ y)
w = np.linalg.solve(X.T @ X, X.T @ y)
r = y - X @ w
print("w = (c, m) =", w, "  residual =", r, "  SSE =", r @ r)
print("Xᵀr (should be 0, residual ⟂ columns):", X.T @ r)
print("np.linalg.lstsq agrees:", np.linalg.lstsq(X, y, rcond=None)[0])

XᵀX =
 [[ 4.  6.]
 [ 6. 14.]] 
Xᵀy = [12. 23.]
w = (c, m) = [1.5 1. ]   residual = [-0.5  0.5  0.5 -0.5]   SSE = 1.0000000000000002
Xᵀr (should be 0, residual ⟂ columns): [-0.  0.]
np.linalg.lstsq agrees: [1.5 1. ]


In [15]:
# GPS-style positioning in 2-D: 4 beacons, noisy ranges, unknown position (true = (3, 4))
P = np.array([[0, 0], [10, 0], [0, 10], [10, 10.]])
r = np.array([5.05, 8.00, 6.75, 9.20])          # measured ranges (true: 5, 8.062, 6.708, 9.220)
# subtract beacon-1 equation from the others → linear in (x, y)
A = 2 * (P[1:] - P[0])
b = r[0]**2 - r[1:]**2 + (P[1:]**2).sum(1) - (P[0]**2).sum()
print("A =\n", A, "\nb =", b)
pos = np.linalg.solve(A.T @ A, A.T @ b)
print("least-squares position:", pos, "  error vs truth:", round(np.linalg.norm(pos - [3, 4]), 4))
print("GDOP-like factor sqrt(trace((AᵀA)⁻¹)) =", round(np.sqrt(np.trace(np.linalg.inv(A.T @ A))), 4))

A =
 [[20.  0.]
 [ 0. 20.]
 [20. 20.]] 
b = [ 61.5025  79.94   140.8625]
least-squares position: [3.0655 3.9873]   error vs truth: 0.0667
GDOP-like factor sqrt(trace((AᵀA)⁻¹)) = 0.0577


## Part 7 — Conditioning

In [16]:
K = np.array([[1, 1], [1, 1.0001]])
for b2 in [2.0001, 2.0002]:
    print(f"b = (2, {b2}) → x =", np.linalg.solve(K, [2, b2]).round(6))
print("cond(K) =", round(np.linalg.cond(K), 1), "  singular values:", np.linalg.svd(K)[1])
rel_b = 1e-4 / np.linalg.norm([2, 2.0001]); rel_x = np.linalg.norm([-1, 1]) / np.linalg.norm([1, 1])
print(f"relative change in b = {rel_b:.3e}, in x = {rel_x:.3f}, amplification = {rel_x/rel_b:.0f} (≤ κ)")

b = (2, 2.0001) → x = [1. 1.]
b = (2, 2.0002) → x = [0. 2.]
cond(K) = 40002.0   singular values: [2.0001 0.    ]
relative change in b = 3.535e-05, in x = 1.000, amplification = 28285 (≤ κ)


In [17]:
# Hilbert matrices: famous ill-conditioned family. Solve Hx = b where the true x is all ones.
from scipy.linalg import hilbert
for n in [4, 8, 12]:
    H = hilbert(n); xt = np.ones(n); b = H @ xt
    xs = np.linalg.solve(H, b); xi = np.linalg.inv(H) @ b
    print(f"n={n:2d} cond={np.linalg.cond(H):9.2e}  err(solve)={np.linalg.norm(xs-xt):.2e}  "
          f"err(inv@b)={np.linalg.norm(xi-xt):.2e}  residual solve={np.linalg.norm(H@xs-b):.1e} inv={np.linalg.norm(H@xi-b):.1e}")

n= 4 cond= 1.55e+04  err(solve)=2.46e-13  err(inv@b)=1.14e-12  residual solve=1.1e-16 inv=3.3e-13
n= 8 cond= 1.53e+10  err(solve)=3.41e-07  err(inv@b)=1.91e-06  residual solve=5.7e-16 inv=9.6e-08
n=12 cond= 1.63e+16  err(solve)=6.21e-01  err(inv@b)=2.01e+01  residual solve=1.1e-16 inv=4.8e-02


## Part 8 — Diet problem as a 3×3 system

In [18]:
# columns = foods (oats, milk, peanut butter); rows = protein, carbs, fat (grams per serving)
N = np.array([[5, 8, 8], [27, 12, 6], [3, 8, 16]])
target = np.array([22, 69, 22])
print("det =", round(np.linalg.det(N)), "→ servings =", np.linalg.solve(N, target))
print("Cramer agrees:", cramer(N, target)[0])

det = -1152 → servings = [2.  1.  0.5]
Cramer agrees: [2.  1.  0.5]


## Part 9 — More case studies: Leontief economy and breaking the Hill cipher

In [19]:
# Leontief input-output model: x = A x + d  =>  (I - A) x = d
# A[i, j] = units of sector i consumed to make one unit of sector j
A_io = np.array([[0.2, 0.3], [0.4, 0.1]])
d = np.array([60.0, 90.0])                   # final (external) demand
IA = np.eye(2) - A_io
print("I - A =\n", IA, "\ndet(I - A) =", round(np.linalg.det(IA), 4))
print("(I - A)⁻¹ =\n", np.linalg.inv(IA))
x_io = np.linalg.solve(IA, d)
print("gross output x =", x_io, "  check (I - A)x =", IA @ x_io)

I - A =
 [[ 0.8 -0.3]
 [-0.4  0.9]] 
det(I - A) = 0.6
(I - A)⁻¹ =
 [[1.5    0.5   ]
 [0.6667 1.3333]]
gross output x = [135. 160.]   check (I - A)x = [60. 90.]


In [20]:
# Known-plaintext attack: the Hill cipher is linear, so a few known pairs give a linear system for K.
# Plaintext pairs as columns P, ciphertext pairs as columns C:  K P = C (mod 26)  =>  K = C P⁻¹ (mod 26)
Pm = sp.Matrix([[7, 11], [4, 15]])     # HE, LP
Cm = sp.Matrix([[7, 0], [8, 19]])      # HI, AT
print("det P =", Pm.det(), "≡", Pm.det() % 26, "(mod 26), invertible since gcd(9, 26) = 1")
print("P⁻¹ mod 26 =", Pm.inv_mod(26).tolist())
print("recovered K =", (Cm * Pm.inv_mod(26)).applyfunc(lambda t: t % 26).tolist())

det P = 61 ≡ 9 (mod 26), invertible since gcd(9, 26) = 1
P⁻¹ mod 26 = [[19, 19], [14, 21]]
recovered K = [[3, 3], [2, 5]]


## Part 10 — Checks for the practice problems in the note

In [21]:
M = sp.Matrix
print("P7  det(triangular) =", M([[2, 5, -1], [0, 3, 7], [0, 0, -4]]).det())
print("P9  det =", M([[1, 2, 3], [4, 5, 6], [7, 8, 10]]).det())
A10 = M([[2, 0, 1], [1, 1, 0], [0, 1, 1]])
print("P10 det =", A10.det(), " Adj =", A10.adjugate().tolist(), " inverse =", A10.inv().tolist())
A11 = M([[1, 1, 1], [2, -1, 1], [1, 2, -1]]); b11 = M([6, 3, 2])
def col_replaced(A, k, b):
    T = A.copy(); T[:, k] = b; return T
print("P11 D =", A11.det(), " D_i =", [col_replaced(A11, k, b11).det() for k in range(3)], " x =", list(A11.solve(b11)))
print("P15 F(30°) =\n", F, " F @ (2,0) =", F @ np.array([2, 0]))
x16 = np.array([1, 2, 3, 4.]); y16 = np.array([2, 3, 5, 6.]); X16 = np.c_[np.ones(4), x16]
w16 = np.linalg.solve(X16.T @ X16, X16.T @ y16); r16 = y16 - X16 @ w16
print("P16 XᵀX =", (X16.T @ X16).tolist(), " Xᵀy =", X16.T @ y16, " w =", w16, " SSE =", round(r16 @ r16, 6))
K = np.array([[3, 3], [2, 5]]); c17 = hill("MATH", K); print("P17 MATH →", c17, "→", hill(c17, inv_mod26(K)))
print("P18 5⁻¹ mod 26 =", pow(5, -1, 26))
print("P19 cond(diag(100, 0.01)) =", np.linalg.cond(np.diag([100, 0.01])), " cond(rotation) =", round(np.linalg.cond(rot(0.7)), 6))
k = sp.symbols("k"); print("P20 det =", sp.expand(M([[1, 2, k], [0, 1, 3], [2, k, 1]]).det()))
Bud = M([[2, 3, 1], [1, 2, 2], [3, 1, 1]]); print("P21 det =", Bud.det(), " prices =", list(Bud.solve(M([360, 490, 370]))))
A23 = M([[1, 1], [0, 1]]); B23 = M([[2, 0], [0, 3]])
print("P23 (AB)⁻¹ =", (A23 * B23).inv().tolist(), " B⁻¹A⁻¹ =", (B23.inv() * A23.inv()).tolist())
print("P24 det(I+I) =", M.eye(2).__add__(M.eye(2)).det(), " vs det I + det I = 2")

P7  det(triangular) = -24
P9  det = -3
P10 det = 3  Adj = [[1, 1, -1], [-1, 2, 1], [1, -2, 2]]  inverse = [[1/3, 1/3, -1/3], [-1/3, 2/3, 1/3], [1/3, -2/3, 2/3]]
P11 D = 7  D_i = [7, 14, 21]  x = [1, 2, 3]
P15 F(30°) =
 [[ 0.5    0.866]
 [ 0.866 -0.5  ]]  F @ (2,0) = [1.     1.7321]
P16 XᵀX = [[4.0, 10.0], [10.0, 30.0]]  Xᵀy = [16. 47.]  w = [0.5 1.4]  SSE = 0.2
P17 MATH → KYAV → MATH
P18 5⁻¹ mod 26 = 21
P19 cond(diag(100, 0.01)) = 10000.0  cond(rotation) = 1.0
P20 det = 13 - 5*k
P21 det = 10  prices = [50, 20, 200]
P23 (AB)⁻¹ = [[1/2, -1/2], [0, 1/3]]  B⁻¹A⁻¹ = [[1/2, -1/2], [0, 1/3]]
P24 det(I+I) = 4  vs det I + det I = 2
